# DA-Fusion Pipeline

Compares three data augmentation strategies using the same model and training setup as the CEMS pipeline:
- **Baseline**: Real data only, DINOv2 features from cache
- **RandAugment**: Real data with classical augmentation applied at DINOv2 extraction time
- **DA-Fusion**: Real + diffusion-generated synthetic images (from `generate_augmented.py`)

All experiments use `BiomassModel` (DINOv2 → 32-d latent → 5 targets) trained with `train_cems`.

## Imports and config

In [1]:
import sys, os
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from PIL import Image
from sklearn.preprocessing import MinMaxScaler

REPO_ROOT = Path(r"C:\Users\Eier\OneDrive\Maskinlæring\INF367A\CSIRO---Image2Biomass-Prediction")
sys.path.insert(0, str(REPO_ROOT / "src"))

from shared.data_utils import make_splits, _make_wide_df, _image_id_from_path, TARGETS
from shared.model import BiomassModel
from shared.train import eval_epoch
from shared.metrics import weighted_global_r2, rmse_per_target
from methods.cems.train_cems import train_cems
from methods.cems.cems_utils import compute_neigh_size, precompute_knn
from methods.cems.estimate_id import estimate_id

CACHE_DIR = REPO_ROOT / "src" / "cache"
AUG_CSV   = REPO_ROOT / "data" / "tabular" / "train" / "train_augmented.csv"
CSV_PATH  = REPO_ROOT / "data" / "tabular" / "train" / "train.csv"
IMAGE_DIR = REPO_ROOT / "data" / "image"

cfg = dict(
    seed=42, epochs=80, lr=3e-4, weight_decay=1e-3,
    latent_dim=32, dropout=0.1, sigma=1e-3, cems_method=1,
)

device = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


## Load cached features and build splits

Reuses the DINOv2 cache from the CEMS pipeline. The same `GroupKFold` split is used across all experiments so results are directly comparable.

In [2]:
# --- Real training images (from CEMS cache) ---
features_np = np.load(CACHE_DIR / "features_dinov2.npy").astype(np.float32)   # (N, 384)
image_ids   = np.load(CACHE_DIR / "image_ids.npy", allow_pickle=True)          # (N,)

df_wide = _make_wide_df(CSV_PATH)
id_to_label = {
    _image_id_from_path(row["image_path"]): row[TARGETS].values.astype(np.float32)
    for _, row in df_wide.iterrows()
}
labels_np = np.stack(
    [id_to_label.get(iid, np.zeros(5, dtype=np.float32)) for iid in image_ids]
)  # (N, 5)

id_to_idx = {iid: i for i, iid in enumerate(image_ids)}
train_ids, val_ids = make_splits(CSV_PATH)

train_idx = [id_to_idx[i] for i in train_ids if i in id_to_idx]
val_idx   = [id_to_idx[i] for i in val_ids   if i in id_to_idx]

X_real_train = features_np[train_idx]   # (N_train, 384)
Y_real_train = labels_np[train_idx]     # (N_train, 5)
X_real_val   = features_np[val_idx]     # (N_val, 384)
Y_real_val   = labels_np[val_idx]       # (N_val, 5)

# Validation dataset — shared across all experiments
val_ds = TensorDataset(
    torch.tensor(X_real_val), torch.tensor(Y_real_val)
)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

print(f"N_train={len(train_idx)}  N_val={len(val_idx)}")
print(f"Real features: {X_real_train.shape}  Labels: {Y_real_train.shape}")

# --- Augmented (synthetic) images ---
AUG_CSV_EXISTS   = AUG_CSV.exists()
AUG_CACHE_EXISTS = (
    (CACHE_DIR / "augmented_features_dinov2.npy").exists() and
    (CACHE_DIR / "augmented_image_ids.npy").exists()
)

if AUG_CACHE_EXISTS and AUG_CSV_EXISTS:
    aug_features_np = np.load(CACHE_DIR / "augmented_features_dinov2.npy").astype(np.float32)
    aug_image_ids   = np.load(CACHE_DIR / "augmented_image_ids.npy", allow_pickle=True)

    df_aug = pd.read_csv(AUG_CSV)
    df_synth = df_aug[df_aug["is_synthetic"].astype(bool)].copy()
    # Map stem (e.g. 'ID123_aug0') → label row
    id_to_label_aug = {
        Path(row["image_path"]).stem: row[TARGETS].values.astype(np.float32)
        for _, row in df_synth.iterrows()
    }
    aug_labels_np = np.stack(
        [id_to_label_aug.get(iid, np.zeros(5, dtype=np.float32)) for iid in aug_image_ids]
    )  # (K, 5)

    print(f"Synthetic features: {aug_features_np.shape}  Labels: {aug_labels_np.shape}")
else:
    aug_features_np = None
    aug_image_ids   = None
    aug_labels_np   = None
    if not AUG_CSV_EXISTS:
        print("[WARN] train_augmented.csv not found. Run generate_augmented.py first.")
    if not AUG_CACHE_EXISTS:
        print("[WARN] Augmented DINOv2 cache not found. Run extract_features.py first.")

N_train=285  N_val=72
Real features: (285, 384)  Labels: (285, 5)
Synthetic features: (1071, 384)  Labels: (1071, 5)


## CEMS setup helper

Shared across all experiments: estimates latent ID, computes neighbourhood size, precomputes kNN.

In [3]:
def make_cems_args(X_train, Y_train, cfg, device, label=""):
    """
    Fit a temporary BiomassModel, estimate latent ID, precompute kNN.
    Returns (cems_args, knn_indices, neigh_size, y_scaler).
    """
    torch.manual_seed(cfg["seed"])
    _tmp = BiomassModel(384, cfg["latent_dim"], 5, cfg["dropout"]).to(device)
    _tmp.eval()
    with torch.no_grad():
        latents = _tmp.encode(
            torch.tensor(X_train, dtype=torch.float32, device=device)
        ).cpu().numpy()
    d_float = estimate_id(latents, label=f"latent {label}")
    d = max(1, int(round(d_float)))
    neigh_size = compute_neigh_size(d)
    print(f"  {label}: d={d_float:.2f} → d={d}  neigh_size={neigh_size}")

    y_scaler = MinMaxScaler()
    y_scaler.fit(Y_train)
    Y_scaled = y_scaler.transform(Y_train).astype(np.float32)

    knn_indices = precompute_knn(X_train, Y_scaled, neigh_size, device=device)

    cems_args = SimpleNamespace(
        sigma       = cfg["sigma"],
        cems_method = cfg["cems_method"],
        id          = d,
        use_hessian = True,
    )
    return cems_args, knn_indices, neigh_size, y_scaler

## Experiment 1: Baseline (real data only)

DINOv2 features from cache → BiomassModel → CEMS training. Reference score.

In [4]:
print("=== EXPERIMENT 1: Baseline (real data only) ===")

cems_args_1, knn_1, neigh_1, scaler_1 = make_cems_args(
    X_real_train, Y_real_train, cfg, device, label="baseline"
)

torch.manual_seed(cfg["seed"])
model_1 = BiomassModel(384, cfg["latent_dim"], 5, cfg["dropout"]).to(device)

history_1 = train_cems(
    model=model_1,
    X_train=X_real_train,
    Y_train_raw=Y_real_train,
    val_ds=val_ds,
    knn_indices=knn_1,
    scaler=scaler_1,
    args=cems_args_1,
    neigh_size=neigh_1,
    epochs=cfg["epochs"],
    lr=cfg["lr"],
    weight_decay=cfg["weight_decay"],
    seed=cfg["seed"],
    device=device,
    verbose=True,
)

_, r2_1, rmse_1, _, _ = eval_epoch(model_1, val_loader, device)
print(f"\nBaseline  val R\u00b2={r2_1:.4f}")
for t in TARGETS:
    print(f"  RMSE {t:<18}: {rmse_1[t]:.4f}")

=== EXPERIMENT 1: Baseline (real data only) ===
  scikit-dimension not found — attempting pip install...
  [latent baseline]  d = 9.80  (method: skdim.TwoNN)
  baseline: d=9.80 → d=10  neigh_size=128
  [train_cems] mode=CEMS-full  neigh_size=128  sigma=0.001  epochs=80
  ep   1  tr=6.6492  real=3.3251  aug=3.3241  val_loss=2.6313  val_R²=0.5287  [Green:18.85  Dead:10.84  Clover:13.85  g:20.34  Total:19.14]
  ep  10  tr=1.5346  real=0.7667  aug=0.7679  val_loss=1.7349  val_R²=0.7867  [Green:12.41  Dead:7.65  Clover:9.32  g:11.98  Total:14.21]
  ep  20  tr=1.2147  real=0.6065  aug=0.6081  val_loss=1.7350  val_R²=0.7986  [Green:12.24  Dead:7.84  Clover:7.66  g:12.52  Total:14.52]
  ep  30  tr=1.0418  real=0.5166  aug=0.5252  val_loss=1.7164  val_R²=0.7928  [Green:11.94  Dead:8.17  Clover:7.56  g:12.74  Total:14.54]
  ep  40  tr=0.9996  real=0.5032  aug=0.4964  val_loss=1.7206  val_R²=0.8025  [Green:11.58  Dead:8.38  Clover:6.99  g:12.62  Total:14.34]
  ep  50  tr=0.9587  real=0.4804  aug=

## Experiment 2: RandAugment baseline

Real images re-extracted with RandAugment applied before DINOv2. Comparison to show classical augmentation vs diffusion-based.

In [7]:
print("=== EXPERIMENT 2: RandAugment baseline ===")

_norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
_transform_randaug = transforms.Compose([
    transforms.Resize((252, 504)),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(), _norm,
])

print("Loading DINOv2 for RandAugment extraction...")
dino_model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", verbose=False)
dino_model.eval().to(device)
for p in dino_model.parameters():
    p.requires_grad_(False)

train_image_id_list = [image_ids[i] for i in train_idx]
id_to_path = {
    _image_id_from_path(row["image_path"]): REPO_ROOT / "data" / "image" / row["image_path"]
    for _, row in df_wide.iterrows()
}

print(f"Extracting RandAugment features for {len(train_image_id_list)} images...")
ra_feats = []
for i, iid in enumerate(train_image_id_list):
    if i % 50 == 0: print(f"  {i}/{len(train_image_id_list)}")
    p = id_to_path.get(iid)
    if p is None or not p.exists():
        ra_feats.append(np.zeros(384, dtype=np.float32))
        continue
    img = Image.open(p).convert("RGB")
    x = _transform_randaug(img).unsqueeze(0).to(device)
    with torch.no_grad():
        ra_feats.append(dino_model(x).squeeze(0).cpu().numpy())
X_ra_train = np.stack(ra_feats).astype(np.float32)
del dino_model

cems_args_2, knn_2, neigh_2, scaler_2 = make_cems_args(
    X_ra_train, Y_real_train, cfg, device, label="randaugment"
)

torch.manual_seed(cfg["seed"])
model_2 = BiomassModel(384, cfg["latent_dim"], 5, cfg["dropout"]).to(device)

history_2 = train_cems(
    model=model_2,
    X_train=X_ra_train,
    Y_train_raw=Y_real_train,
    val_ds=val_ds,
    knn_indices=knn_2,
    scaler=scaler_2,
    args=cems_args_2,
    neigh_size=neigh_2,
    epochs=cfg["epochs"],
    lr=cfg["lr"],
    weight_decay=cfg["weight_decay"],
    seed=cfg["seed"],
    device=device,
    verbose=True,
)

_, r2_2, rmse_2, _, _ = eval_epoch(model_2, val_loader, device)
print(f"\nRandAugment  val R²={r2_2:.4f}")
for t in TARGETS:
    print(f"  RMSE {t:<18}: {rmse_2[t]:.4f}")

=== EXPERIMENT 2: RandAugment baseline ===
Loading DINOv2 for RandAugment extraction...
Extracting RandAugment features for 285 images...
  0/285
  50/285
  100/285
  150/285
  200/285
  250/285
  [latent randaugment]  d = 11.20  (method: skdim.TwoNN)
  randaugment: d=11.20 → d=11  neigh_size=128
  [train_cems] mode=CEMS-full  neigh_size=128  sigma=0.001  epochs=80
  ep   1  tr=6.7152  real=3.3574  aug=3.3578  val_loss=2.4619  val_R²=0.5322  [Green:19.31  Dead:10.93  Clover:14.11  g:20.44  Total:17.14]
  ep  10  tr=1.5129  real=0.7590  aug=0.7539  val_loss=1.6612  val_R²=0.7956  [Green:12.39  Dead:7.86  Clover:9.23  g:11.71  Total:13.75]
  ep  20  tr=1.1634  real=0.5818  aug=0.5816  val_loss=1.7964  val_R²=0.7757  [Green:12.91  Dead:7.84  Clover:8.65  g:13.31  Total:14.91]
  ep  30  tr=1.0298  real=0.5135  aug=0.5163  val_loss=1.7651  val_R²=0.7566  [Green:13.41  Dead:7.86  Clover:8.08  g:13.74  Total:14.51]
  ep  40  tr=0.9437  real=0.4733  aug=0.4704  val_loss=1.7485  val_R²=0.7682  

## Experiment 3: DA-Fusion (real + synthetic)

Combines real training features with DINOv2 features from DA-Fusion synthetic images.
kNN is precomputed over the combined training set. Validation is on real images only.

**Requires**: run `generate_augmented.py` then `extract_features.py` to produce the augmented cache.

In [8]:
if not AUG_CACHE_EXISTS or not AUG_CSV_EXISTS:
    print("[SKIP] Augmented cache not available. See warnings above.")
else:
    print("=== EXPERIMENT 3: DA-Fusion (real + synthetic) ===")
    print(f"  Real train:      {len(X_real_train)}")
    print(f"  Synthetic train: {len(aug_features_np)}")

    # Combine real train + synthetic for training
    X_daf_train = np.vstack([X_real_train, aug_features_np])   # (N_train + K, 384)
    Y_daf_train = np.vstack([Y_real_train, aug_labels_np])     # (N_train + K, 5)

    cems_args_3, knn_3, neigh_3, scaler_3 = make_cems_args(
        X_daf_train, Y_daf_train, cfg, device, label="da-fusion"
    )

    torch.manual_seed(cfg["seed"])
    model_3 = BiomassModel(384, cfg["latent_dim"], 5, cfg["dropout"]).to(device)

    history_3 = train_cems(
        model=model_3,
        X_train=X_daf_train,
        Y_train_raw=Y_daf_train,
        val_ds=val_ds,
        knn_indices=knn_3,
        scaler=scaler_3,
        args=cems_args_3,
        neigh_size=neigh_3,
        epochs=cfg["epochs"],
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
        seed=cfg["seed"],
        device=device,
        verbose=True,
    )

    _, r2_3, rmse_3, _, _ = eval_epoch(model_3, val_loader, device)
    print(f"\nDA-Fusion  val R\u00b2={r2_3:.4f}")
    for t in TARGETS:
        print(f"  RMSE {t:<18}: {rmse_3[t]:.4f}")

=== EXPERIMENT 3: DA-Fusion (real + synthetic) ===
  Real train:      285
  Synthetic train: 1071
  [latent da-fusion]  d = 14.13  (method: skdim.TwoNN)
  da-fusion: d=14.13 → d=14  neigh_size=128
  [train_cems] mode=CEMS-full  neigh_size=128  sigma=0.001  epochs=80
  ep   1  tr=5.2186  real=2.6072  aug=2.6115  val_loss=2.3683  val_R²=0.6563  [Green:16.97  Dead:11.46  Clover:10.24  g:16.40  Total:17.98]
  ep  10  tr=1.6076  real=0.8066  aug=0.8010  val_loss=1.7991  val_R²=0.8175  [Green:10.31  Dead:9.31  Clover:8.79  g:10.96  Total:14.64]
  ep  20  tr=1.2561  real=0.6285  aug=0.6275  val_loss=1.7398  val_R²=0.8292  [Green:11.61  Dead:8.12  Clover:7.60  g:12.11  Total:13.78]
  ep  30  tr=1.0861  real=0.5429  aug=0.5432  val_loss=1.7771  val_R²=0.8137  [Green:11.28  Dead:9.29  Clover:8.77  g:12.25  Total:13.81]
  ep  40  tr=1.0145  real=0.5069  aug=0.5076  val_loss=1.6930  val_R²=0.8240  [Green:11.29  Dead:9.12  Clover:7.71  g:12.06  Total:13.18]
  ep  50  tr=0.9418  real=0.4724  aug=0.4

## Results Summary

In [9]:
header = f"{'Method':<16} | {'Val R\u00b2':>6} | " + " | ".join(f"{t.split('_')[1]:>8}" for t in TARGETS)
sep = "-" * len(header)
print(sep)
print(header)
print(sep)

rows = [("Baseline", r2_1, rmse_1), ("RandAugment", r2_2, rmse_2)]
if AUG_CACHE_EXISTS and AUG_CSV_EXISTS:
    rows.append(("DA-Fusion", r2_3, rmse_3))

for name, r2, rmse in rows:
    rmse_vals = " | ".join(f"{rmse[t]:>8.2f}" for t in TARGETS)
    print(f"{name:<16} | {r2:>6.4f} | {rmse_vals}")
print(sep)

--------------------------------------------------------------------------------
Method           | Val R² |    Green |     Dead |   Clover |        g |    Total
--------------------------------------------------------------------------------
Baseline         | 0.7998 |    11.75 |     8.15 |     6.89 |    12.81 |    14.45
RandAugment      | 0.7568 |    13.81 |     7.84 |     8.33 |    13.89 |    14.18
DA-Fusion        | 0.8259 |    12.14 |     9.12 |     7.55 |    12.51 |    12.96
--------------------------------------------------------------------------------
